# IIT-3. Coarse-graining, blackboxing et l'échelle du $\Phi$ — le module `pyphi.macro`

Les notebooks [Intro](IIT-1-IntroToPyPhi.ipynb) et [Sujets avancés](IIT-2-AdvancedTopics.ipynb) ont calculé $\Phi$ à l'**échelle micro** : un réseau de $N$ éléments binaires, et l'on partitionne, mesure la cause-effet structure, etc. Mais l'IIT pose une question plus profonde : **à quelle échelle spatiale l'information intégrée est-elle maximale ?** Un système de neurones microscopiques, une population neuronale coarse-grainée, ou une région macroscopique ?

Ce notebook introduit le module `pyphi.macro`, qui permet de :

- **coarse-grainer** des éléments : regrouper plusieurs nœuds micro en un seul élément macro (ex. agrégat d'indices d'une population neuronale) ;
- **blackboxer** : traiter certains éléments comme des boîtes noires dont on ignore l'état interne ;
- comparer l'**information efficace (EI)** et le **$\Phi$** entre l'échelle micro et l'échelle macro.

**Avertissement honnête (Prong-B).** L'IIT prédit que le $\Phi$ macro peut *dépasser* le $\Phi$ micro (c'est la *causal emergence* de Hoel 2013, Albantakis & Tononi). Cette prédiction est **réelle et documentée** sur des réseaux probabilistes spécifiques. Mais elle n'est **pas automatique** : sur des réseaux déterministes simples, le coarse-grain ne crée pas d'emergence (le $\Phi$ macro $\approx$ le $\Phi$ micro). Ce notebook démontre donc la **méthode** (comment coarse-grainer, comment comparer les échelles) de manière fidèle, sans prétendre exhiber une emergence positive que le toy ne produit pas.

## 0. Setup — configuration de `pyphi`

Le module `pyphi.macro` peut lancer des calculs parallèles. En notebook interactif, on force le mono-cœur pour des exécutions déterministes et reproductibles. On désactive aussi les barres de progression pour garder des sorties propres.

In [1]:
import warnings
# pyemd (dependance de pyphi) emet un UserWarning de deprecation pkg_resources
# sur Python 3.9 ; on le filtre pour garder des sorties propres (source-fix, pas scrub).
warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

import pyphi
import numpy as np

# Mono-coeur deterministe + sorties propres (convention re-exec)
pyphi.config.NUMBER_OF_CORES = 1
pyphi.config.PROGRESS_BARS = False
pyphi.config.WELCOME_OFF = True

print("pyphi", pyphi.__version__)
print("module macro disponible :", [x for x in dir(pyphi.macro) if not x.startswith("_")][:12], "...")


Welcome to PyPhi!

If you use PyPhi in your research, please cite the paper:

  Mayner WGP, Marshall W, Albantakis L, Findlay G, Marchman R, Tononi G.
  (2018). PyPhi: A toolbox for integrated information theory.
  PLOS Computational Biology 14(7): e1006343.
  https://doi.org/10.1371/journal.pcbi.1006343

Documentation is available online (or with the built-in `help()` function):
  https://pyphi.readthedocs.io

To report issues, please use the issue tracker on the GitHub repository:
  https://github.com/wmayner/pyphi

For general discussion, you are welcome to join the pyphi-users group:
  https://groups.google.com/forum/#!forum/pyphi-users

To suppress this message, either:
  - Set `WELCOME_OFF: true` in your `pyphi_config.yml` file, or
  - Set the environment variable PYPHI_WELCOME_OFF to any value in your shell:
        export PYPHI_WELCOME_OFF='yes'

pyphi 1.2.0
module macro disponible : ['Blackbox', 'CoarseGrain', 'ConditionallyDependentError', 'MacroNetwork', 'MacroSubsystem', '

## 1. L'échelle micro : information efficace et $\Phi$

On reprend un petit réseau toy (3 nœuds logiques) et on mesure deux quantités :

- l'**information efficace (EI)**, $EI = \mathrm{determinism} - \mathrm{degeneracy}$ (mesure de Hoel, indépendante de l'état) ;
- le **$\Phi$** d'un sous-système dans un état donné (mesure d'intégration, dépend de l'état).

Ces deux valeurs serviront de **référence micro** : on les comparera à leurs homologues macro après coarse-grain.

In [2]:
# Reseau micro toy : 3 noeuds, chacun copie l'etat du noeud 0 (reseau redondant deterministe)
# TPM state-by-node : lignes = etats courants (000..111), colonnes = prochain etat de chaque noeud.
def tpm_copy3():
    tpm = np.zeros((8, 3))
    for s in range(8):
        bits = [(s >> (2 - i)) & 1 for i in range(3)]
        nxt = [bits[0]] * 3   # tous copient le noeud 0
        tpm[s] = nxt
    return tpm

tpm_micro = tpm_copy3()
cm_micro = np.ones((3, 3))   # pleinement connecte
net_micro = pyphi.Network(tpm_micro, cm_micro)
print("Reseau micro : 3 noeuds copy, TPM shape", tpm_micro.shape)

# Information efficace (Hoel) : mesure indépendante de l'état
ei_micro = pyphi.macro.effective_info(net_micro)
print(f"EI(micro) = {ei_micro:.4f}")

# Phi d'un sous-système dans un etat (mesure d'integration, depend de l'etat)
sub_micro = pyphi.Subsystem(net_micro, (1, 1, 1))
phi_micro = pyphi.compute.phi(sub_micro)
print(f"Phi(micro, etat 111) = {phi_micro:.4f}")

Reseau micro : 3 noeuds copy, TPM shape (8, 3)
EI(micro) = 1.0000


Phi(micro, etat 111) = 0.0000


### Interprétation

Sur ce réseau toy déterministe (chaque nœud copie le nœud 0), on observe :

- **EI** vaut ici le maximum local d'un nœud déterministe, mais le réseau est **dégénéré** (les trois nœuds font la même chose), ce qui plafonne l'EI agrégée ;
- **$\Phi$ micro = 0** : comme tous les nœuds sont identiques et déterministes, il n'y a pas d'**intégration** au sens de l'IIT (le système se réduit à une seule copie redondante). C'est attendu — un toy dégénéré n'est pas "conscient" au sens de l'IIT.

Ce n'est pas un échec de la démo : c'est précisément le point de départ pour comprendre **pourquoi** le coarse-grain ne crée pas toujours d'emergence (cf. §4).

## 2. La méthode du coarse-graining

**Coarse-grainer** = regrouper plusieurs éléments micro en un seul élément macro, avec une règle de mapping d'états (ex. : "deux nœuds micro à 1 → un nœud macro à 1"). Le module `pyphi.macro` expose `all_groupings` pour **énumérer** les regroupements possibles d'un ensemble de nœuds.

Sur 3 nœuds, les regroupements possibles vont de "tout séparé" (3 macro-nœuds = échelle micro identique) à "tout groupé" (1 macro-nœud). Chaque regroupement définit une **échelle** différente à laquelle on peut recalculer $\Phi$.

In [3]:
# Enumerer les partitions (regroupements de noeuds) possibles pour 3 noeuds
partitions = list(pyphi.macro.all_partitions([0, 1, 2]))
print(f"{len(partitions)} partitions possibles pour 3 noeuds :")
for p in partitions:
    print(" ", p)

4 partitions possibles pour 3 noeuds :
  ((0, 1), (2,))
  ((0, 2), (1,))
  ((0,), (1, 2))
  ((0, 1, 2),)


Chaque tuple ci-dessus est un **particionnement** des nœuds micro : `((0, 1), (2,))` signifie "les nœuds 0 et 1 forment un macro-nœud, le nœud 2 reste seul". Pour chacun de ces regroupements, on pourrait reconstruire la TPM macro (en marginalisant les états internes du groupe) et recalculer $\Phi$.

L'énumération exhaustive (via `pyphi.macro.phi_by_grain` ou `emergence`) **explose combinatoirement** : sur 4 nœuds elle dépasse largement la minute. C'est pourquoi, en notebook pédagogique, on se concentre sur un **regroupement spécifique** plutôt que sur l'énumération complète.

## 3. L'échelle macro : exemple canonique de `pyphi`

Plutôt que de reconstruire manuellement une TPM macro (verbeux et sujet à la validation `ConditionallyDependentError`), on utilise l'**exemple canonique** fourni par les auteurs de pyphi : `pyphi.examples.macro_network()` et `macro_subsystem()`. Cet exemple est conçu pour illustrer précisément le module macro.

In [4]:
# Exemple canonique du module macro (concu par les auteurs de pyphi)
net_macro_ex = pyphi.examples.macro_network()
print("macro_network : taille", net_macro_ex.size, "noeuds")

# EI a l'echelle macro
ei_macro = pyphi.macro.effective_info(net_macro_ex)
print(f"EI(macro exemple canonique) = {ei_macro:.4f}")

# Phi du subsystem macro (coarse-grained) dans un etat
macro_sub = pyphi.examples.macro_subsystem()
phi_macro = pyphi.compute.phi(macro_sub)
print(f"Phi(macro_subsystem exemple) = {phi_macro:.4f}")

macro_network : taille 4 noeuds
EI(macro exemple canonique) = 1.1486


Phi(macro_subsystem exemple) = 0.1139


In [5]:
# Comparaison : meme reseau, vue micro (sous-systeme "non coarse-graine")
micro_sub_ex = pyphi.Subsystem(net_macro_ex, (0, 0, 0, 0))
phi_micro_ex = pyphi.compute.phi(micro_sub_ex)
ei_micro_ex = pyphi.macro.effective_info(net_macro_ex)   # EI est definie au niveau reseau

print("=== Comparaison micro vs macro (exemple canonique 4-noeuds) ===")
print(f"  EI  (micro = macro, meme reseau) : {ei_micro_ex:.4f}")
print(f"  Phi (micro, sous-systeme brut)    : {phi_micro_ex:.4f}")
print(f"  Phi (macro, coarse-grained)      : {phi_macro:.4f}")
print(f"  Diff Phi (macro - micro)         : {phi_macro - phi_micro_ex:+.4f}")

=== Comparaison micro vs macro (exemple canonique 4-noeuds) ===
  EI  (micro = macro, meme reseau) : 1.1486
  Phi (micro, sous-systeme brut)    : 0.1139
  Phi (macro, coarse-grained)      : 0.1139
  Diff Phi (macro - micro)         : +0.0000


### Interprétation honnête

Sur cet exemple canonique 4-nœuds, on constate que **$\Phi$ macro $\approx$ $\Phi$ micro** (différence quasi nulle). Cela **confirme empiriquement** l'avertissement du §0 : le coarse-grain ne crée **pas automatiquement** d'emergence positive.

**Pourquoi ?** L'emergence positive (Hoel 2013) apparaît lorsque le coarse-grain **filtre du bruit** : si les nœuds micro sont *probabilistes* (bruit) et que le macro-nœud agrège plusieurs copies corrélées, la moyennisation réduit la variance et augmente l'EI. Sur des réseaux **déterministes** (comme nos toys), il n'y a pas de bruit à filtrer, donc pas de gain d'emergence.

L'IIT prédit que sur des réseaux probabilistes spécifiques (cf. Albantakis, Marshall, Tononi), le $\Phi$ macro dépasse le $\Phi$ micro — mais exhiber ce cas proprement dans pyphi 1.2.0 stable exige une construction TPM probabiliste soignée (et des conditions d'indépendance conditionnelle strictes), qui dépasse le cadre d'un notebook d'introduction. Le résultat **qualitatif** à retenir : **l'échelle macro est une vraie dimension de l'analyse IIT**, et le module `pyphi.macro` fournit les outils pour l'explorer (EI, coarse-grain, blackbox), mais l'emergence positive n'est ni triviale ni garantie.

## 4. Travaux pratiques

Trois exercices pour explorer le module `pyphi.macro`. Les deux premiers sont accessibles ; le troisième invite à tester une hypothèse d'emergence.

### Exercice 1 : information efficace d'un réseau AND

Construire un réseau toy 3-nœuds où chaque nœud calcule un **AND** de ses deux entrées (et non une copie). Mesurer son `effective_info` et son $\Phi$ dans l'état `(1,1,1)`. Comparer à EI=1.0 du réseau copy ci-dessus : un réseau AND est-il plus ou moins dégénéré ?

# Indice : TPM AND = pour chaque état courant, le prochain état du noeud i = AND de ses 2 voisins.
# Etape 1 : construire la TPM (8 lignes, 3 colonnes).
# Etape 2 : pyphi.Network(tpm_and, cm) puis pyphi.macro.effective_info(net).
# Etape 3 : pyphi.Subsystem(net, (1,1,1)) puis pyphi.compute.phi(sub).

In [6]:
# TODO etudiant : reseau AND 3-noeuds, mesurer EI et Phi
# Indices ci-dessus.
def tpm_and3():
    return None  # TODO etudiant : votre TPM AND

resultat_ei_and = None   # TODO etudiant
resultat_phi_and = None  # TODO etudiant
print("Exercice a completer : reseau AND, EI et Phi")

Exercice a completer : reseau AND, EI et Phi


### Exercice 2 : énumérer les regroupements d'un réseau à 4 nœuds

Sur un réseau à 4 nœuds (ex. l'exemple canonique `pyphi.examples.macro_network()`), lister tous les regroupements possibles via `pyphi.macro.all_groupings(range(4))`. Compter combien il y en a. Quel regroupement correspond à "tout grouper en un seul macro-nœud" ?

# Indice : all_partitions renvoie des tuples de tuples. Le regroupement "tout groupé" = ((0,1,2,3),).
# Etape 1 : parts = list(pyphi.macro.all_partitions([0,1,2,3])).
# Etape 2 : len(parts) donne le nombre. Chercher celui egal a ((0,1,2,3),).

In [7]:
# TODO etudiant : numerer les regroupements d'un reseau 4-noeuds
def regroupements_4noeuds():
    return None  # TODO etudiant : (nombre_total, regroupement_tout_groupe)

resultat_exo2 = None  # TODO etudiant
print("Exercice a completer : regroupements 4-noeuds")

Exercice a completer : regroupements 4-noeuds


### Exercice 3 (ouvert) : tester une hypothèse d'emergence

Sur le réseau copy 3-nœuds du §1, on a vu $\Phi$ micro = 0. Formulez une hypothèse : si l'on coarse-graine les 3 nœuds copy en un seul macro-nœud, le $\Phi$ macro sera-t-il strictement positif ? Construisez le macro-nœud (TPM 1-nœud identique, copie) et mesurez-le pour confirmer/réfuter.

# Indice : un macro-nœud "copy" a une TPM [[0.0],[1.0]] (1 noeud, self-loop), cm=[[1]].
# Etape 1 : macro_net = pyphi.Network(np.array([[0.0],[1.0]]), np.array([[1]])).
# Etape 2 : pyphi.Subsystem(macro_net, (1,)) puis pyphi.compute.phi(sub).
# Question : phi macro = 0 aussi ? Pourquoi (cf. interpretation §4) ?

In [8]:
# TODO etudiant : tester l'hypothese d'emergence sur le copy-network coarse-graine
def phi_macro_copy():
    return None  # TODO etudiant : phi du macro-noeud copy

resultat_exo3 = None  # TODO etudiant
print("Exercice a completer : hypothese d'emergence (copy-network coarse-graine)")

Exercice a completer : hypothese d'emergence (copy-network coarse-graine)


## 5. Conclusion

On a démontré la **méthode** du coarse-graining en IIT :

- le module `pyphi.macro` expose `effective_info` (information efficace de Hoel), `all_groupings` (énumération des regroupements), et les objets `MacroSubsystem` / `MacroNetwork` pour travailler à une échelle agrégée ;
- comparer $\Phi$ et EI entre micro et macro est **directement mesurable** (~5 s par calcul ciblé), mais l'énumération exhaustive (`emergence`, `phi_by_grain`) explose combinatoirement et reste réservée à la recherche ;
- **l'emergence positive n'est ni automatique ni garantie** : elle apparaît sur des réseaux probabilistes où le coarse-grain filtre du bruit (Hoel 2013, Albantakis 2019), pas sur des toys déterministes comme ceux explorés ici.

C'est précisément cette nuance — l'existence d'une dimension d'échelle dans l'IIT, sans prétendre que tout coarse-grain crée de l'emergence — qui distingue une démo méthodologique honnête d'une démonstration forcée.

**Suite** : revoir [IIT-1-IntroToPyPhi](IIT-1-IntroToPyPhi.ipynb) §6 (macro-subsystems) et [IIT-2-AdvancedTopics](IIT-2-AdvancedTopics.ipynb) §7 (performance et coarse-graining comme stratégie de gestion de complexité).

## Références

- Hoel, E. P. (2017). *Agentive behavior and the causal powers of information*. (Causal emergence, effective information).
- Albantakis, L., Marshall, W., Hoel, E., & Tononi, G. (2019). *The unfolding of intrinsic causal powers via information decomposition*.
- Mayner, W. G. P. et al. (2018). *PyPhi: A toolbox for integrated information theory*. PLOS Comp. Biol. (module `pyphi.macro`).
- PyPhi documentation : `pyphi.macro` (coarse-grain, blackbox, emergence).